In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
file_path = "../data/personalized_healthcare_synthetic_dataset.xlsx"

df_original = pd.read_excel(
    file_path,
    sheet_name="Synthetic_Patient_Data"
)
df = df_original.copy()

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully
Dataset shape: (500, 15)


,patient_id,age_years,sex,condition,glucose_mg_dL,systolic_bp_mmHg,diastolic_bp_mmHg,cholesterol_mg_dL,heart_rate_bpm,recorded_allergy,family_history,adherence_level,historical_medication_class,recorded_outcome,dataset_notice
0,SYN-0001,28,Female,Seasonal Allergy,97,132,85,214,73,None recorded,No,High,Antihistamine class A,Improved,Synthetic educational record; not medical advice
1,SYN-0002,80,Female,Acid Reflux,80,123,79,189,59,None recorded,Yes,Medium,Acid-suppression class A,Improved,Synthetic educational record; not medical advice
2,SYN-0003,36,Female,Asthma,86,121,66,192,82,None recorded,No,Medium,Controller inhaler class,Follow-up required,Synthetic educational record; not medical advice
3,SYN-0004,21,Male,High Cholesterol,103,120,69,278,85,None recorded,Yes,High,Lipid-lowering class B,Follow-up required,Synthetic educational record; not medical advice
4,SYN-0005,58,Male,High Cholesterol,79,111,76,244,64,None recorded,Yes,High,Lipid-lowering class B,Follow-up required,Synthetic educational record; not medical advice


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Number of rows: 500
Number of columns: 15

Column names:
['patient_id', 'age_years', 'sex', 'condition', 'glucose_mg_dL', 'systolic_bp_mmHg', 'diastolic_bp_mmHg', 'cholesterol_mg_dL', 'heart_rate_bpm', 'recorded_allergy', 'family_history', 'adherence_level', 'historical_medication_class', 'recorded_outcome', 'dataset_notice']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   patient_id                   500 non-null    object
 1   age_years                    500 non-null    int64 
 2   sex                          500 non-null    object
 3   condition                    500 non-null    object
 4   glucose_mg_dL                500 non-null    int64 
 5   systolic_bp_mmHg             500 non-null    int64 
 6   diastolic_bp_mmHg            500 non-null    int64 
 7   cholesterol_mg_dL          

In [4]:
quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2),
    "unique_values": df.nunique()
})

quality_report

,data_type,missing_values,missing_percentage,unique_values
patient_id,object,0,0.0,500
age_years,int64,0,0.0,63
sex,object,0,0.0,2
condition,object,0,0.0,6
glucose_mg_dL,int64,0,0.0,107
systolic_bp_mmHg,int64,0,0.0,76
diastolic_bp_mmHg,int64,0,0.0,51
cholesterol_mg_dL,int64,0,0.0,130
heart_rate_bpm,int64,0,0.0,39
recorded_allergy,object,0,0.0,5


In [5]:
duplicate_rows = df.duplicated().sum()
duplicate_patient_ids = df["patient_id"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate patient IDs:", duplicate_patient_ids)

Duplicate rows: 0
Duplicate patient IDs: 0


In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print(df.columns.tolist())

['patient_id', 'age_years', 'sex', 'condition', 'glucose_mg_dl', 'systolic_bp_mmhg', 'diastolic_bp_mmhg', 'cholesterol_mg_dl', 'heart_rate_bpm', 'recorded_allergy', 'family_history', 'adherence_level', 'historical_medication_class', 'recorded_outcome', 'dataset_notice']


In [7]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text columns cleaned successfully")

Text columns cleaned successfully


In [8]:
missing_labels = [
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "unknown"
]

for column in text_columns:

    missing_mask = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(missing_labels)
    )

    df.loc[missing_mask, column] = np.nan

print("Missing-value labels standardized")
print(df.isnull().sum())

Missing-value labels standardized
patient_id                     0
age_years                      0
sex                            0
condition                      0
glucose_mg_dl                  0
systolic_bp_mmhg               0
diastolic_bp_mmhg              0
cholesterol_mg_dl              0
heart_rate_bpm                 0
recorded_allergy               0
family_history                 0
adherence_level                0
historical_medication_class    0
recorded_outcome               0
dataset_notice                 0
dtype: int64


In [9]:
df["patient_id_valid"] = df["patient_id"].str.match(
    r"^SYN-\d{4}$",
    na=False
)

invalid_patient_ids = df.loc[
    ~df["patient_id_valid"],
    ["patient_id"]
]

print("Invalid Patient IDs:", len(invalid_patient_ids))
invalid_patient_ids.head()

Invalid Patient IDs: 0


,patient_id


In [10]:
numeric_columns = [
    "age_years",
    "glucose_mg_dl",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "cholesterol_mg_dl",
    "heart_rate_bpm"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print(df[numeric_columns].dtypes)

age_years            int64
glucose_mg_dl        int64
systolic_bp_mmhg     int64
diastolic_bp_mmhg    int64
cholesterol_mg_dl    int64
heart_rate_bpm       int64
dtype: object


In [11]:
valid_ranges = {
    "age_years": (0, 120),
    "glucose_mg_dl": (20, 700),
    "systolic_bp_mmhg": (50, 300),
    "diastolic_bp_mmhg": (30, 200),
    "cholesterol_mg_dl": (50, 700),
    "heart_rate_bpm": (20, 250)
}

range_check_results = []

for column, (minimum, maximum) in valid_ranges.items():

    invalid_count = (
        (df[column] < minimum) |
        (df[column] > maximum)
    ).sum()

    range_check_results.append({
        "column": column,
        "minimum_allowed": minimum,
        "maximum_allowed": maximum,
        "observed_minimum": df[column].min(),
        "observed_maximum": df[column].max(),
        "invalid_values": invalid_count
    })

range_check_report = pd.DataFrame(range_check_results)
range_check_report

,column,minimum_allowed,maximum_allowed,observed_minimum,observed_maximum,invalid_values
0,age_years,0,120,18,80,0
1,glucose_mg_dl,20,700,72,225,0
2,systolic_bp_mmhg,50,300,103,189,0
3,diastolic_bp_mmhg,30,200,65,118,0
4,cholesterol_mg_dl,50,700,146,330,0
5,heart_rate_bpm,20,250,58,96,0


In [12]:
allowed_categories = {
    "sex": ["Male", "Female"],

    "condition": [
        "Seasonal Allergy",
        "Type 2 Diabetes",
        "Hypertension",
        "Asthma",
        "Acid Reflux",
        "High Cholesterol"
    ],

    "recorded_allergy": [
        "None recorded",
        "Sulfonamide",
        "Penicillin",
        "Latex",
        "NSAID"
    ],

    "family_history": ["Yes", "No"],

    "adherence_level": [
        "Low",
        "Medium",
        "High"
    ],

    "recorded_outcome": [
        "Improved",
        "Stable",
        "Follow-up required"
    ]
}

category_check_results = []

for column, allowed_values in allowed_categories.items():
    invalid_mask = ~df[column].isin(allowed_values) & df[column].notna()

    category_check_results.append({
        "column": column,
        "invalid_categories": invalid_mask.sum(),
        "invalid_values": df.loc[
            invalid_mask, column
        ].unique().tolist()
    })

category_check_report = pd.DataFrame(category_check_results)
category_check_report

,column,invalid_categories,invalid_values
0,sex,0,[]
1,condition,0,[]
2,recorded_allergy,0,[]
3,family_history,0,[]
4,adherence_level,0,[]
5,recorded_outcome,0,[]


In [13]:
for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - (1.5 * iqr)
    upper_limit = q3 + (1.5 * iqr)

    df[f"{column}_outlier"] = (
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    )

outlier_columns = [
    column for column in df.columns
    if column.endswith("_outlier")
]

df[outlier_columns].sum()

age_years_outlier             0
glucose_mg_dl_outlier        63
systolic_bp_mmhg_outlier     26
diastolic_bp_mmhg_outlier    12
cholesterol_mg_dl_outlier    22
heart_rate_bpm_outlier        0
dtype: int64

In [14]:
df["has_missing_data"] = df.isnull().any(axis=1)

df["has_numeric_outlier"] = df[
    outlier_columns
].any(axis=1)

df["data_quality_status"] = np.where(
    df["has_missing_data"],
    "Review missing data",
    np.where(
        ~df["patient_id_valid"],
        "Review patient ID",
        np.where(
            df["has_numeric_outlier"],
            "Review possible outlier",
            "Valid"
        )
    )
)

df["data_quality_status"].value_counts()

data_quality_status
Valid                      381
Review possible outlier    119
Name: count, dtype: int64

In [15]:
original_clean_columns = (
    df_original.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .tolist()
)
base_columns = [
    column for column in original_clean_columns
    if column in df.columns
]
duplicate_rows_count = df.duplicated(
    subset=base_columns
).sum()
missing_values_count = df[base_columns].isnull().sum().sum()

final_quality_summary = pd.DataFrame({
    "metric": [
        "Original rows",
        "Cleaned rows",
        "Original columns",
        "Missing values",
        "Duplicate rows",
        "Duplicate patient IDs",
        "Invalid patient IDs",
        "Rows requiring review"
    ],
    "value": [
        len(df_original),
        len(df),
        df_original.shape[1],
        missing_values_count,
        duplicate_rows_count,
        df["patient_id"].duplicated().sum(),
        (~df["patient_id_valid"]).sum(),
        (df["data_quality_status"] != "Valid").sum()
    ]
})

final_quality_summary

,metric,value
0,Original rows,500
1,Cleaned rows,500
2,Original columns,15
3,Missing values,0
4,Duplicate rows,0
5,Duplicate patient IDs,0
6,Invalid patient IDs,0
7,Rows requiring review,119


In [16]:
columns_to_remove = [
    "dataset_notice",
    "patient_id_valid",
    "has_missing_data",
    "has_numeric_outlier"
] + outlier_columns

cleaned_ml_data = df.drop(
    columns=columns_to_remove,
    errors="ignore"
).copy()

print("Cleaned ML dataset shape:", cleaned_ml_data.shape)
cleaned_ml_data.head()

Cleaned ML dataset shape: (500, 15)


,patient_id,age_years,sex,condition,glucose_mg_dl,systolic_bp_mmhg,diastolic_bp_mmhg,cholesterol_mg_dl,heart_rate_bpm,recorded_allergy,family_history,adherence_level,historical_medication_class,recorded_outcome,data_quality_status
0,SYN-0001,28,Female,Seasonal Allergy,97,132,85,214,73,None recorded,No,High,Antihistamine class A,Improved,Valid
1,SYN-0002,80,Female,Acid Reflux,80,123,79,189,59,None recorded,Yes,Medium,Acid-suppression class A,Improved,Valid
2,SYN-0003,36,Female,Asthma,86,121,66,192,82,None recorded,No,Medium,Controller inhaler class,Follow-up required,Valid
3,SYN-0004,21,Male,High Cholesterol,103,120,69,278,85,None recorded,Yes,High,Lipid-lowering class B,Follow-up required,Valid
4,SYN-0005,58,Male,High Cholesterol,79,111,76,244,64,None recorded,Yes,High,Lipid-lowering class B,Follow-up required,Valid


In [18]:
review_records = df.loc[
    df["data_quality_status"] != "Valid"
].copy()

print("Records requiring review:", len(review_records))

output_file = "cleaned_personalized_healthcare_data.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    df_original.to_excel(
        writer,
        sheet_name="Original_Data",
        index=False
    )

    cleaned_ml_data.to_excel(
        writer,
        sheet_name="Cleaned_Data",
        index=False
    )

    quality_report.to_excel(
        writer,
        sheet_name="Quality_Report",
        index=True
    )

    range_check_report.to_excel(
        writer,
        sheet_name="Range_Check",
        index=False
    )

    category_check_report.to_excel(
        writer,
        sheet_name="Category_Check",
        index=False
    )

    final_quality_summary.to_excel(
        writer,
        sheet_name="Quality_Summary",
        index=False
    )

    review_records.to_excel(
        writer,
        sheet_name="Review_Records",
        index=False
    )

print("Cleaned dataset saved as:", output_file)

Records requiring review: 119
Cleaned dataset saved as: cleaned_personalized_healthcare_data.xlsx
